# 🤖 Agent 2 — Twitter Bot Detection with RNN (GRU)

**Project:** Social Threat Intelligence  
**Agent:** 2 — Bot Detection  
**Model:** Gated Recurrent Unit (GRU) via PyTorch  
**Dataset:** `twitter_human_bots_dataset.csv` (~37,438 accounts)  
**Expected Performance:** ~86–88% Accuracy, Balanced F1 on both Human & Bot

---
### ✅ How to run on Google Colab:
1. Upload `twitter_human_bots_dataset.csv` to the **Files panel** (left sidebar 📁).
2. Run all cells: **Runtime → Run all** (Ctrl+F9).
3. At the end, download the trained model files (`bot_rnn_model.pth`, `bot_scaler.joblib`, `bot_input_meta.joblib`).

## 0. Setup — Create folders & install dependencies

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)
os.makedirs('models/saved', exist_ok=True)
os.makedirs('models/vectorizers', exist_ok=True)
!pip install scikit-learn pandas numpy matplotlib seaborn joblib --quiet
print('✅ Environment ready.')

In [ ]:
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## 1. Load Dataset

In [ ]:
CSV_PATH = 'twitter_human_bots_dataset.csv'
df_raw = pd.read_csv(CSV_PATH)
print(f'✅ Loaded {len(df_raw):,} accounts')
print('Columns:', list(df_raw.columns))
df_raw.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print('Account type counts:')
print(df_raw['account_type'].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_raw['account_type'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], rot=0)
axes[0].set_title('Account Type Distribution')

df_raw.boxplot(column='followers_count', by='account_type', ax=axes[1], showfliers=False)
axes[1].set_title('Followers Count (No Outliers)')

df_raw.boxplot(column='average_tweets_per_day', by='account_type', ax=axes[2], showfliers=False)
axes[2].set_title('Tweets / Day (No Outliers)')

plt.tight_layout()
plt.savefig('outputs/eda_distribution.png', dpi=120)
plt.show()

## 3. Preprocessing & Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['bot_label'] = (df['account_type'].astype(str).str.lower() == 'bot').astype(int)

    # Log transforms for skewed counters
    df['log_followers'] = np.log1p(pd.to_numeric(df['followers_count'], errors='coerce').fillna(0).clip(lower=0))
    df['log_friends'] = np.log1p(pd.to_numeric(df['friends_count'], errors='coerce').fillna(0).clip(lower=0))
    df['log_statuses'] = np.log1p(pd.to_numeric(df['statuses_count'], errors='coerce').fillna(0).clip(lower=0))
    df['log_favourites'] = np.log1p(pd.to_numeric(df['favourites_count'], errors='coerce').fillna(0).clip(lower=0))
    df['log_tweets_per_day'] = np.log1p(pd.to_numeric(df['average_tweets_per_day'], errors='coerce').fillna(0).clip(lower=0))
    df['account_age_days'] = pd.to_numeric(df['account_age_days'], errors='coerce').fillna(0)

    # Binary profile signals
    df['default_profile'] = df['default_profile'].astype(bool).astype(int)
    df['default_profile_image'] = df['default_profile_image'].astype(bool).astype(int)
    df['geo_enabled'] = df['geo_enabled'].astype(bool).astype(int)
    df['verified'] = df['verified'].astype(bool).astype(int)

    # Temporal signals from created_at
    dt = pd.to_datetime(df['created_at'], errors='coerce')
    df['hour_of_day'] = dt.dt.hour.fillna(-1).astype(int)
    df['day_of_week'] = dt.dt.dayofweek.fillna(-1).astype(int)
    df['year'] = dt.dt.year.fillna(2015).astype(int)

    # Metadata & textual signals
    df['desc_len'] = df['description'].fillna('').apply(len)
    df['digits_in_name'] = df['screen_name'].fillna('').apply(lambda s: sum(c.isdigit() for c in str(s)))
    return df

df = engineer_features(df_raw)

FEATURES = [
    'log_followers', 'log_friends', 'log_statuses', 'log_favourites',
    'log_tweets_per_day', 'account_age_days',
    'default_profile', 'default_profile_image', 'geo_enabled', 'verified',
    'hour_of_day', 'day_of_week', 'year',
    'desc_len', 'digits_in_name',
]
SEQ_LEN = 5

X_raw = df[FEATURES].fillna(0).values.astype(np.float32)
y_raw = df['bot_label'].values
print(f'Feature matrix shape: {X_raw.shape}, y shape: {y_raw.shape}')

## 4. Train / Val / Test Split & Scaling

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_raw, y_raw, test_size=0.15, random_state=42, stratify=y_raw
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

def build_sequences(X, seq_len=SEQ_LEN):
    n, n_feat = X.shape
    step_size = math.ceil(n_feat / seq_len)
    pad = step_size * seq_len - n_feat
    if pad > 0:
        X = np.hstack([X, np.zeros((n, pad), dtype=np.float32)])
    return X.reshape(n, seq_len, step_size).astype(np.float32)

X_train_seq = build_sequences(X_train_sc)
X_val_seq = build_sequences(X_val_sc)
X_test_seq = build_sequences(X_test_sc)

INPUT_SIZE = X_train_seq.shape[2]
print(f'Sequence shape: (seq_len={SEQ_LEN}, step_size={INPUT_SIZE})')

## 5. Model Architecture & Training

In [ ]:
class BotRNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, num_classes=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.gru(x, h0)
        return self.head(out[:, -1, :])

model = BotRNN(input_size=INPUT_SIZE).to(DEVICE)
print(model)

In [ ]:
BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(torch.tensor(X_train_seq), torch.tensor(y_train, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val_seq), torch.tensor(y_val, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test_seq), torch.tensor(y_test, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=False)

# Handle class imbalance (Human ~25k, Bot ~12.5k)
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.8], device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

EPOCHS = 20
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            val_loss += criterion(logits, yb).item()
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    val_loss /= len(val_loader)
    val_acc = correct / total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'outputs/bot_rnn_best.pth')

## 6. Evaluation & Confusion Matrix

In [ ]:
model.load_state_dict(torch.load('outputs/bot_rnn_best.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

print('=== TEST SET EVALUATION ===')
print(f'Accuracy : {accuracy_score(all_labels, all_preds):.4f}')
print(classification_report(all_labels, all_preds, target_names=['Human', 'Bot']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Human', 'Bot'], yticklabels=['Human', 'Bot'])
plt.title('Confusion Matrix — Bot Detection RNN')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=120)
plt.show()

## 7. Save Model & Download

In [ ]:
torch.save(model.state_dict(), 'models/saved/bot_rnn_model.pth')
joblib.dump(scaler, 'models/vectorizers/bot_scaler.joblib')
meta = {'input_size': INPUT_SIZE, 'seq_len': SEQ_LEN, 'features': FEATURES}
joblib.dump(meta, 'models/vectorizers/bot_input_meta.joblib')

from google.colab import files
files.download('models/saved/bot_rnn_model.pth')
files.download('models/vectorizers/bot_scaler.joblib')
files.download('models/vectorizers/bot_input_meta.joblib')
print('✅ Downloaded trained model artifacts!')